# KV260 için YOLOX-Nano × VisDrone Fine-Tune (Kaggle)

Bu not defteri, KV260 DPU'suna uyumlu (**ReLU + DPUFocus**) YOLOX-Nano modelini **VisDrone2019-DET** veri setiyle eğitir ve Vitis AI kuantalaması için gerekli çıktıları paketler.

## Çalıştırmadan önce (Kaggle ayarları)
1. **Settings → Accelerator**: GPU (T4 veya P100) seçin.
2. **Settings → Internet**: **On** yapın (repo klonlama ve ağırlık indirme için).
3. (Önerilen) Kaggle'da `VisDrone2019-DET` araması yapıp uygun bir dataset'i **+ Add Input** ile bağlayın; `images/` ve `annotations/` klasörleri olan herhangi bir kopya otomatik bulunur. Bulunamazsa not defteri resmi Google Drive bağlantılarından indirmeyi dener.

## Akış
kurulum → VisDrone edinme → COCO'ya dönüştürme → fine-tune (~80 epoch) → resmi DET tarzı AP@500 değerlendirme → görsel kontrol (merkez noktalarıyla) → `artifacts.zip`

## Kota notları
- Tek oturum ~12 saat, haftalık GPU kotası ~30 saattir.
- Eğitim yarıda kalırsa: bu not defterinin çıktısını (Output) yeni oturuma **input olarak bağlayın** ve aşağıdaki *Devam (resume)* hücresini kullanın.

In [ ]:
import os
from pathlib import Path

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
%cd {WORK}
!nvidia-smi

In [ ]:
# YOLOX kurulumu (Kaggle'daki hazir torch surumune dokunmadan).
# Kaggle ve Vitis AI VM ayni commit'i kullanir; main dali kullanilmaz.
import importlib
import site
import subprocess
import sys

YOLOX_COMMIT = "6ddff4824372906469a7fae2dc3206c7aa4bbaee"
YOLOX_DIR = Path(WORK) / "YOLOX"

if not YOLOX_DIR.is_dir():
    !git clone --filter=blob:none --no-checkout https://github.com/Megvii-BaseDetection/YOLOX.git "{YOLOX_DIR}"
!git -C "{YOLOX_DIR}" fetch --depth 1 origin {YOLOX_COMMIT}
!git -C "{YOLOX_DIR}" checkout --detach {YOLOX_COMMIT}
current = !git -C "{YOLOX_DIR}" rev-parse HEAD
assert current and current[0] == YOLOX_COMMIT, f"Yanlis YOLOX commit'i: {current}"


def _pip(*args):
    """pip'i cagirir. `!pip` kabuk cagrisinin aksine hatayi yutmaz."""
    proc = subprocess.run([sys.executable, "-m", "pip", *args],
                          capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout[-4000:])
        print(proc.stderr[-4000:])
    return proc.returncode


# --no-build-isolation sart: YOLOX'un setup.py'si torch'u import eder, pip'in
# izole build ortaminda torch bulunmaz ve kurulum sessizce basarisiz olur.
rc = _pip("install", "--no-deps", "--no-build-isolation", "-e", str(YOLOX_DIR))
if rc != 0:
    print("Editable kurulum basarisiz; editable olmayan kuruluma dusuluyor.")
    rc = _pip("install", "--no-deps", "--no-build-isolation", str(YOLOX_DIR))
assert rc == 0, "YOLOX kurulumu basarisiz (yukaridaki pip ciktisina bakin)."

assert _pip("install", "-q", "loguru", "tabulate", "psutil", "pycocotools",
            "thop", "ninja", "gdown") == 0, "Yardimci paket kurulumu basarisiz."

# pip'in yazdigi .pth dosyalari yalnizca yorumlayici acilisinda okunur; calisan
# kernel'in sys.path'ini elle tazelemezsek import ayni oturumda basarisiz olur.
for _site_dir in getattr(site, "getsitepackages", list)():
    site.addsitedir(_site_dir)
if str(YOLOX_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOX_DIR))
importlib.invalidate_caches()

# numpy 1.24+ uyumlulugu: kaldirilan eski takma adlar icin shim
import numpy as np
for _alias, _type in (("float", float), ("int", int), ("bool", bool)):
    if _alias not in np.__dict__:
        setattr(np, _alias, _type)

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
import yolox
print("yolox:", yolox.__version__, "|", yolox.__file__)


In [ ]:
%%writefile visdrone2coco.py
#!/usr/bin/env python3
"""VisDrone-DET etiketlerini COCO JSON formatina donusturur.

VisDrone satir formati:
    <bbox_left>,<bbox_top>,<bbox_width>,<bbox_height>,<score>,<category>,<truncation>,<occlusion>

Kategori esleme:
    0  = ignored-region  -> image.ignore_regions alaninda korunur
    1..10                -> COCO category_id 1..10 (asagidaki sinif listesi)
    11 = others          -> resmi protokol gibi sinif hedeflerinden cikartilir
    score == 0, cat 1..10 -> kendi sinifinda COCO crowd/ignore kutusu

Egitim loader'i ignored-region ve score=0 alanlarini 114 ile maskeler.
Degerlendirici, category=0 bolgesinin en az yuzde 50'si icinde kalan tespitleri
resmi VisDrone dropObjectsInIgr adimina uygun olarak sonuclardan cikartir.

Kullanim:
    python visdrone2coco.py --image-dir VisDrone2019-DET-train/images \
        --anno-dir VisDrone2019-DET-train/annotations \
        --output datasets/visdrone_coco/annotations/instances_train.json
"""

import argparse
import json
from pathlib import Path

from PIL import Image

try:
    from tqdm import tqdm
except ImportError:  # tqdm yoksa sade dongu
    def tqdm(iterable, **kwargs):
        return iterable

VISDRONE_CLASSES = (
    "pedestrian", "people", "bicycle", "car", "van",
    "truck", "tricycle", "awning-tricycle", "bus", "motor",
)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

#: --classes 2: VisDrone kaynakli saha tanimi. VisDrone yalnizca kara
#: tasiti icerdigi icin sinif adi bilerek `land_vehicle`: deniz araci
#: ayri bir veri setinden ucuncu sinif olarak eklenecek.
#: Ignore ve ignored-region mantigi degismez; yalnizca kimlikler eslenir.
TWO_CLASS_NAMES = ("person", "land_vehicle")
TWO_CLASS_MAP = {
    1: 1, 2: 1,                                  # pedestrian, people
    3: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 2, 9: 2, 10: 2,  # tum kara tasitlari
}

SCHEMES = {
    "10": (VISDRONE_CLASSES, {i: i for i in range(1, 11)}),
    "2": (TWO_CLASS_NAMES, TWO_CLASS_MAP),
}


def covered_fraction_xywh(box, regions):
    """Kutu alaninin ignore dikdortgenleri birlesimi icindeki orani."""
    x, y, w, h = (float(v) for v in box)
    clipped = []
    for rx, ry, rw, rh in regions:
        left, top = max(x, rx), max(y, ry)
        right, bottom = min(x + w, rx + rw), min(y + h, ry + rh)
        if right > left and bottom > top:
            clipped.append((left, top, right, bottom))
    if w <= 0 or h <= 0 or not clipped:
        return 0.0
    xs = sorted({edge for rect in clipped for edge in (rect[0], rect[2])})
    covered = 0.0
    for left, right in zip(xs, xs[1:]):
        intervals = sorted(
            (top, bottom)
            for x1, top, x2, bottom in clipped
            if x1 < right and x2 > left
        )
        if not intervals:
            continue
        union_y = 0.0
        start, end = intervals[0]
        for next_start, next_end in intervals[1:]:
            if next_start > end:
                union_y += end - start
                start, end = next_start, next_end
            else:
                end = max(end, next_end)
        covered += (right - left) * (union_y + end - start)
    return covered / (w * h)


def convert(image_dir, anno_dir, output, classes="10"):
    if classes not in SCHEMES:
        raise SystemExit(f"HATA: bilinmeyen sinif semasi {classes!r}")
    class_names, class_map = SCHEMES[classes]

    image_dir, anno_dir, output = Path(image_dir), Path(anno_dir), Path(output)
    img_files = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMG_EXTS)
    if not img_files:
        raise SystemExit(f"HATA: {image_dir} icinde goruntu bulunamadi")

    images, annotations = [], []
    ann_id = 1
    per_class = {name: 0 for name in class_names}
    skipped = 0
    global_ignored = 0
    class_ignored = 0
    dropped_in_ignore = 0

    for img_id, img_path in enumerate(tqdm(img_files, desc="donusturuluyor"), start=1):
        # PIL yalnizca basligi okur, tum goruntuyu cozmez (hizli)
        with Image.open(img_path) as im:
            width, height = im.size
        image_record = {
            "id": img_id,
            "file_name": img_path.name,
            "width": width,
            "height": height,
            "ignore_regions": [],
        }
        images.append(image_record)
        image_ann_start = len(annotations)

        txt = anno_dir / (img_path.stem + ".txt")
        if not txt.exists():
            continue
        for line in txt.read_text().splitlines():
            line = line.strip().rstrip(",")
            if not line:
                continue
            parts = line.split(",")
            if len(parts) < 6:
                skipped += 1
                continue
            x, y, w, h, score, cat = (int(float(v)) for v in parts[:6])
            # goruntu sinirlarina kirp
            x2 = min(x + w, width)
            y2 = min(y + h, height)
            x = max(x, 0)
            y = max(y, 0)
            w = x2 - x
            h = y2 - y
            if w <= 0 or h <= 0:
                skipped += 1
                continue
            if cat == 0:
                image_record["ignore_regions"].append([x, y, w, h])
                global_ignored += 1
                continue
            if cat == 11 or cat < 1 or cat > 10:
                skipped += 1
                continue
            category_id = class_map[cat]
            if score == 0:
                annotations.append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": category_id,
                    "bbox": [x, y, w, h],
                    "area": w * h,
                    "iscrowd": 1,
                    "ignore": 1,
                })
                ann_id += 1
                class_ignored += 1
                continue
            annotations.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": category_id,
                "bbox": [x, y, w, h],
                "area": w * h,
                "iscrowd": 0,
            })
            per_class[class_names[category_id - 1]] += 1
            ann_id += 1

        if image_record["ignore_regions"]:
            image_annotations = annotations[image_ann_start:]
            kept = [
                annotation for annotation in image_annotations
                if covered_fraction_xywh(
                    annotation["bbox"], image_record["ignore_regions"]
                ) < 0.5
            ]
            dropped_in_ignore += len(image_annotations) - len(kept)
            annotations[image_ann_start:] = kept

    per_class = {name: 0 for name in class_names}
    for annotation in annotations:
        if not annotation.get("iscrowd", 0):
            per_class[class_names[annotation["category_id"] - 1]] += 1

    coco = {
        "info": {
            "description": "VisDrone2019-DET (COCO formati)",
            "class_scheme": classes,
        },
        "images": images,
        "annotations": annotations,
        "categories": [
            {"id": i + 1, "name": name, "supercategory": "none"}
            for i, name in enumerate(class_names)
        ],
    }
    output.parent.mkdir(parents=True, exist_ok=True)
    with open(output, "w") as f:
        json.dump(coco, f)

    print(f"\nYazildi: {output}")
    print(f"  goruntu : {len(images)}")
    print(f"  kutu    : {len(annotations)} (atlanan: {skipped})")
    print(f"  ignore  : {global_ignored} genel bolge, {class_ignored} sinif-ozel kutu")
    print(f"  maskede : {dropped_in_ignore} hedef resmi protokole gore cikartildi")
    for name, count in per_class.items():
        print(f"    {name:16s} {count}")
    return coco


def parse_args():
    p = argparse.ArgumentParser(
        description="VisDrone-DET etiketlerini COCO JSON formatina cevirir"
    )
    p.add_argument("--image-dir", required=True, help="VisDrone images/ klasoru")
    p.add_argument("--anno-dir", required=True, help="VisDrone annotations/ klasoru")
    p.add_argument("--output", required=True, help="Cikti COCO JSON yolu")
    p.add_argument(
        "--classes", default="10", choices=sorted(SCHEMES),
        help="10 = resmi VisDrone siniflari, 2 = person/vehicle saha tanimi",
    )
    return p.parse_args()


if __name__ == "__main__":
    args = parse_args()
    convert(args.image_dir, args.anno_dir, args.output, classes=args.classes)


In [ ]:
# VisDrone2019-DET edinme: once bagli Kaggle dataset'lerini tara, yoksa indir
import zipfile


def find_visdrone_split(split):
    name = f"VisDrone2019-DET-{split}"
    roots = [Path("/kaggle/input"), Path(WORK) / "VisDrone"]
    patterns = [name, f"*/{name}", f"*/*/{name}", f"*/*/*/{name}"]
    for root in roots:
        if not root.exists():
            continue
        for pat in patterns:
            for cand in root.glob(pat):
                if (cand / "images").is_dir() and (cand / "annotations").is_dir():
                    return cand.resolve()
    return None


# Resmi Google Drive dosyalari (github.com/VisDrone/VisDrone-Dataset)
GDRIVE_IDS = {
    "train": "1a2oHjcEcwXP8oUF95qiwrqzACb2YlUhn",  # ~1.44 GB
    "val": "1bxK5zgLn0_L8x276eKkuYA_FzwCIjb59",    # ~0.07 GB
}

TRAIN_SRC = find_visdrone_split("train")
VAL_SRC = find_visdrone_split("val")

if TRAIN_SRC is None or VAL_SRC is None:
    print("Bagli dataset bulunamadi, Google Drive'dan indiriliyor...")
    import gdown

    dl_dir = Path(WORK) / "VisDrone"
    dl_dir.mkdir(exist_ok=True)
    for split, fid in GDRIVE_IDS.items():
        zip_path = dl_dir / f"VisDrone2019-DET-{split}.zip"
        if not zip_path.exists():
            gdown.download(id=fid, output=str(zip_path), quiet=False)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(dl_dir)
    TRAIN_SRC = find_visdrone_split("train")
    VAL_SRC = find_visdrone_split("val")

assert TRAIN_SRC and VAL_SRC, (
    "VisDrone bulunamadi! Kaggle'da 'VisDrone2019-DET' arayip bir dataset'i "
    "input olarak baglayin veya Internet erisimini acin."
)
print("train:", TRAIN_SRC)
print("val  :", VAL_SRC)

In [ ]:
# COCO formatina donustur + goruntu klasorlerini symlink'le
DATASET_DIR = Path(WORK) / "datasets" / "visdrone_coco"
(DATASET_DIR / "annotations").mkdir(parents=True, exist_ok=True)

for link_name, src in (("train_images", TRAIN_SRC / "images"),
                       ("val_images", VAL_SRC / "images")):
    link = DATASET_DIR / link_name
    if not link.exists():
        os.symlink(src, link, target_is_directory=True)

# --classes 2 -> person / vehicle saha tanimi (bkz. exps dosyasi)
CLASS_SCHEME = "2"
!python visdrone2coco.py --classes {CLASS_SCHEME} --image-dir "{TRAIN_SRC}/images" --anno-dir "{TRAIN_SRC}/annotations" --output "{DATASET_DIR}/annotations/instances_train.json"
!python visdrone2coco.py --classes {CLASS_SCHEME} --image-dir "{VAL_SRC}/images" --anno-dir "{VAL_SRC}/annotations" --output "{DATASET_DIR}/annotations/instances_val.json"

In [ ]:
%%writefile visdrone_eval.py
#!/usr/bin/env python3
"""VisDrone DET toolkit'inin AP@500 protokolunun saf Python karsiligi.

AP hesabi resmi `calcAccuracy.m` ile birebir ayni: ignore GT'ler recall
paydasinda kalir (`rec = tp/max(1,numel(gtMatch))`).

Ek olarak, resmi protokolde bulunmayan iki tamamlayici cikti uretilir:
  * P/R/F1 (IoU 0.50) - sabit esikte ve en iyi F1 noktasinda. Yayinlanmis
    YOLO calismalariyla kiyaslanabilmesi icin bu metrigin recall paydasi
    ignore GT'leri **icermez**; AP'ninkinden farklidir, bilincli.
  * Sinif gruplama - 10 ince sinifi 3 saha sinifina indirger. Eslestirme
    hem GT'ye hem tespitlere uygulanir, yani `van`-`car` karisikligi
    degerlendirme asamasinda ortadan kalkar; model yeniden egitilmez.
"""

from collections import defaultdict

import numpy as np

VISDRONE_CLASSES = (
    "pedestrian", "people", "bicycle", "car", "van",
    "truck", "tricycle", "awning-tricycle", "bus", "motor",
)

#: 10 ince VisDrone sinifi -> 3 saha sinifi. Sahada ayrilmasi anlamli olan
#: gruplar korunur; 20 pikselde insanin bile ayiramadigi ayrimlar birlesir.
GROUP_3 = {
    1: 1, 2: 1,                  # pedestrian, people      -> person
    3: 2, 7: 2, 8: 2, 10: 2,     # bicycle, tricycle,
                                 # awning-tricycle, motor  -> two/three-wheeler
    4: 3, 5: 3, 6: 3, 9: 3,      # car, van, truck, bus    -> vehicle
}
GROUP_3_NAMES = {1: "person", 2: "twowheeler", 3: "vehicle"}

#: Hedef saha tanimi: person + vehicle. `vehicle` her turlu tasiti kapsar
#: (bisiklet ve motosiklet dahil). Egitim de bu tanimla yapilir.
GROUP_2 = {
    1: 1, 2: 1,                                   # pedestrian, people -> person
    3: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 2, 9: 2, 10: 2,   # geri kalani -> vehicle
}
GROUP_2_NAMES = {1: "person", 2: "vehicle"}

#: Tek sinifli tanimlar. Eslenmeyen kategoriler hem GT'den hem tespitlerden
#: **dusurulur** (gruplanmaz): sistem o nesneleri hedeflemiyorsa onlari
#: bulamamak da bulmak da puanlanmamalidir.
PERSON_ONLY = {1: 1, 2: 1}
VEHICLE_ONLY = {k: 1 for k in (3, 4, 5, 6, 7, 8, 9, 10)}

#: Degerlendirme senaryolari: ad -> (kategori eslemesi, sinif adlari).
#: `None` esleme resmi 10 sinifli protokol demektir.
SCENARIOS = {
    "10 sinif (resmi)": (None, None),
    "2 sinif (hedef)": (GROUP_2, GROUP_2_NAMES),
    "3 sinif (ara)": (GROUP_3, GROUP_3_NAMES),
    "tek sinif: person": (PERSON_ONLY, {1: "person"}),
    "tek sinif: vehicle": (VEHICLE_ONLY, {1: "vehicle"}),
}

#: En iyi F1 aramasinda taranan guven esikleri.
SCORE_GRID = np.round(np.arange(0.0, 1.0001, 0.01), 4)


def covered_fraction_xywh(box, regions):
    """Bir xywh kutusunun ignore dikdortgenleri birlesimi icindeki oranini bulur."""
    x, y, w, h = (float(v) for v in box)
    if w <= 0 or h <= 0:
        return 0.0
    clipped = []
    for rx, ry, rw, rh in regions:
        left, top = max(x, rx), max(y, ry)
        right, bottom = min(x + w, rx + rw), min(y + h, ry + rh)
        if right > left and bottom > top:
            clipped.append((left, top, right, bottom))
    if not clipped:
        return 0.0

    # Kesisen dikdortgenlerin birlesim alanini x-sweep ile cift saymadan hesapla.
    xs = sorted({edge for rect in clipped for edge in (rect[0], rect[2])})
    covered = 0.0
    for left, right in zip(xs, xs[1:]):
        if right <= left:
            continue
        intervals = sorted(
            (top, bottom)
            for x1, top, x2, bottom in clipped
            if x1 < right and x2 > left
        )
        union_y = 0.0
        if intervals:
            start, end = intervals[0]
            for next_start, next_end in intervals[1:]:
                if next_start > end:
                    union_y += end - start
                    start, end = next_start, next_end
                else:
                    end = max(end, next_end)
            union_y += end - start
        covered += (right - left) * union_y
    return covered / (w * h)


def _overlap(dt, gt, gt_ignore):
    dx, dy, dw, dh = dt
    gx, gy, gw, gh = gt
    iw = min(dx + dw, gx + gw) - max(dx, gx)
    ih = min(dy + dh, gy + gh) - max(dy, gy)
    if iw <= 0 or ih <= 0:
        return 0.0
    inter = iw * ih
    if gt_ignore:
        return inter / max(dw * dh, 1e-12)
    union = dw * dh + gw * gh - inter
    return inter / max(union, 1e-12)


def _eval_image(gt_rows, detections, threshold):
    """VisDrone toolkit evalRes.m ile ayni eslestirme sirasi."""
    # Normal GT once gelir; ignore GT en sona gelir ve birden cok kez eslesebilir.
    gt_rows = sorted(gt_rows, key=lambda row: bool(row["ignore"]))
    gt_state = [-1 if row["ignore"] else 0 for row in gt_rows]
    detections = sorted(detections, key=lambda row: -row["score"])
    dt_matches = []

    for detection in detections:
        best_overlap = threshold
        best_index = None
        best_match = 0
        for index, (ground_truth, state) in enumerate(zip(gt_rows, gt_state)):
            if state == 1:
                continue
            if best_match != 0 and state == -1:
                break
            overlap = _overlap(
                detection["bbox"], ground_truth["bbox"], state == -1
            )
            if overlap < best_overlap:
                continue
            best_overlap = overlap
            best_index = index
            best_match = 1 if state == 0 else -1
        if best_index is not None and best_match == 1:
            gt_state[best_index] = 1
        dt_matches.append((detection["score"], best_match))
    return gt_state, dt_matches


def _voc_ap(recall, precision):
    recall = np.concatenate(([0.0], recall, [1.0]))
    precision = np.concatenate(([0.0], precision, [0.0]))
    for index in range(precision.size - 2, -1, -1):
        precision[index] = max(precision[index], precision[index + 1])
    changes = np.where(recall[1:] != recall[:-1])[0] + 1
    return float(
        np.sum((recall[changes] - recall[changes - 1]) * precision[changes])
    )


def prepare_detections(coco, detections, max_dets=500):
    """Global ignore filtresi ve goruntu basina global top-500 uygular."""
    grouped = defaultdict(list)
    for detection in detections:
        image_id = int(detection["image_id"])
        regions = coco.imgs[image_id].get("ignore_regions", ())
        if covered_fraction_xywh(detection["bbox"], regions) >= 0.5:
            continue
        grouped[image_id].append(detection)

    prepared = {}
    for image_id in coco.getImgIds():
        rows = grouped.get(image_id, ())
        prepared[image_id] = sorted(
            rows,
            key=lambda row: (
                -float(row["score"]),
                int(row["category_id"]),
                tuple(float(v) for v in row["bbox"]),
            ),
        )[:max_dets]
    return prepared


def _f1_curve(dt_rows, real_gt_count):
    """Skor esigi izgarasi uzerinde precision/recall/F1 dizileri uretir.

    `dt_rows` skora gore azalan sirali (skor, eslesme) ciftleridir; eslesme
    1=TP, 0=FP, -1=ignore GT ile eslesti (ne TP ne FP sayilir).

    Recall paydasi **ignore olmayan** GT sayisidir. AP'nin paydasindan
    (resmi toolkit ignore'lari da sayar) bilincli olarak farklidir: bu metrik
    yayinlanmis YOLO sonuclariyla kiyaslanabilsin diye COCO/Ultralytics
    kuralini izler.
    """
    zeros = np.zeros_like(SCORE_GRID)
    if real_gt_count <= 0:
        return zeros, zeros, zeros

    scores = np.asarray([row[0] for row in dt_rows], dtype=np.float64)
    matches = np.asarray([row[1] for row in dt_rows], dtype=np.int8)
    # scores azalan sirali -> -scores artan; "skor >= t" sayisi searchsorted ile
    kept = np.searchsorted(-scores, -SCORE_GRID, side="right")
    tp_cum = np.concatenate(([0.0], np.cumsum(matches == 1, dtype=np.float64)))
    fp_cum = np.concatenate(([0.0], np.cumsum(matches == 0, dtype=np.float64)))
    tp, fp = tp_cum[kept], fp_cum[kept]

    precision = tp / np.maximum(1e-12, tp + fp)
    recall = tp / float(real_gt_count)
    f1 = 2 * precision * recall / np.maximum(1e-12, precision + recall)
    return precision, recall, f1


def _at_grid(index, precision, recall, f1):
    return {
        "precision": float(precision[index]),
        "recall": float(recall[index]),
        "f1": float(f1[index]),
        "score": float(SCORE_GRID[index]),
    }


def names_from_coco(coco, fallback_count=10):
    """Sinif adlarini COCO kategorilerinden okur.

    Boylece veri semasi degistiginde (10 sinif -> 2 sinif) tablolar
    kendiliginden dogru kalir; ad listesi ikinci bir yerde tekrarlanmaz.
    """
    cats = getattr(coco, "cats", None)
    if cats:
        return {int(k): str(v.get("name", k)) for k, v in cats.items()}
    return {i + 1: n for i, n in enumerate(VISDRONE_CLASSES[:fallback_count])}


def evaluate_visdrone(coco, detections, max_dets=500, group_map=None,
                      class_names=None, score_thr=0.30, num_classes=None):
    """AP@[.50:.95], AP50, AP75 ve tamamlayici P/R/F1 degerlerini hesaplar.

    `group_map` verilirse (ornegin `GROUP_3`), sinif kimlikleri hem GT'de hem
    tespitlerde eslenerek degerlendirme gruplanmis siniflar uzerinde yapilir.
    Model degismez; yalnizca olcum degisir.
    """
    prepared = prepare_detections(coco, detections, max_dets=max_dets)
    if num_classes is None:
        num_classes = len(getattr(coco, "cats", None) or VISDRONE_CLASSES)

    def mapped(category_id):
        if group_map is None:
            return category_id if 1 <= category_id <= num_classes else None
        return group_map.get(category_id)

    gt_by_image_class = defaultdict(list)
    dt_by_image_class = defaultdict(list)
    real_gt_count = defaultdict(int)
    available_classes = set()

    for image_id in coco.getImgIds():
        regions = coco.imgs[image_id].get("ignore_regions", ())
        for annotation in coco.imgToAnns.get(image_id, ()):
            category_id = mapped(int(annotation["category_id"]))
            if category_id is None:
                continue
            if covered_fraction_xywh(annotation["bbox"], regions) >= 0.5:
                continue
            ignore = bool(
                annotation.get("ignore", 0) or annotation.get("iscrowd", 0)
            )
            gt_by_image_class[(image_id, category_id)].append({
                "bbox": [float(v) for v in annotation["bbox"]],
                "ignore": ignore,
            })
            if not ignore:
                real_gt_count[category_id] += 1
            available_classes.add(category_id)
        # Tespitleri sinifa gore bir kez ayir: esik dongusunde tekrarlanmasin.
        for row in prepared[image_id]:
            category_id = mapped(int(row["category_id"]))
            if category_id is not None:
                dt_by_image_class[(image_id, category_id)].append(row)

    thresholds = np.arange(0.50, 0.951, 0.05)
    image_ids = coco.getImgIds()
    per_class = {}
    iou50_matches = {}
    for category_id in sorted(available_classes):
        aps = []
        for index, threshold in enumerate(thresholds):
            gt_matches = []
            dt_matches = []
            for image_id in image_ids:
                gt_state, image_dt = _eval_image(
                    gt_by_image_class.get((image_id, category_id), ()),
                    dt_by_image_class.get((image_id, category_id), ()),
                    threshold,
                )
                gt_matches.extend(gt_state)
                dt_matches.extend(image_dt)

            dt_matches.sort(key=lambda row: -row[0])
            if index == 0:  # IoU 0.50 -> F1 egrisi buradan cikar
                iou50_matches[category_id] = dt_matches
            matches = np.asarray([row[1] for row in dt_matches], dtype=np.int8)
            tp = np.cumsum(matches == 1, dtype=np.float64)
            fp = np.cumsum(matches == 0, dtype=np.float64)
            recall = tp / max(1, len(gt_matches))
            precision = tp / np.maximum(1.0, tp + fp)
            aps.append(_voc_ap(recall, precision))
        per_class[category_id] = aps

    names = dict(class_names or {})
    if not names:
        if group_map is None:
            names = names_from_coco(coco, num_classes)
        elif group_map == GROUP_2:
            names = dict(GROUP_2_NAMES)
        elif group_map == GROUP_3:
            names = dict(GROUP_3_NAMES)
        else:
            # Bilinmeyen esleme: adi kaynak sinif adlarindan turet ki tablo
            # "grup 1" gibi anlamsiz bir etiket gostermesin.
            members = defaultdict(list)
            for source, target in sorted(group_map.items()):
                members[target].append(VISDRONE_CLASSES[source - 1])
            names = {t: "+".join(v) for t, v in members.items()}

    empty = {
        "ap": 0.0, "ap50": 0.0, "ap75": 0.0, "per_class": {},
        "per_class_ap50": {}, "class_names": names,
        "f1_best": None, "f1_at": None, "per_class_f1_best": {},
    }
    if not per_class:
        return empty

    curves = {
        category_id: _f1_curve(
            iou50_matches[category_id], real_gt_count[category_id]
        )
        for category_id in sorted(per_class)
    }
    precision_matrix = np.asarray([c[0] for c in curves.values()])
    recall_matrix = np.asarray([c[1] for c in curves.values()])
    f1_matrix = np.asarray([c[2] for c in curves.values()])
    macro = (
        precision_matrix.mean(axis=0),
        recall_matrix.mean(axis=0),
        f1_matrix.mean(axis=0),
    )
    best_index = int(np.argmax(macro[2]))
    fixed_index = int(np.argmin(np.abs(SCORE_GRID - score_thr)))

    matrix = np.asarray(list(per_class.values()), dtype=np.float64)
    return {
        "ap": float(matrix.mean()),
        "ap50": float(matrix[:, 0].mean()),
        "ap75": float(matrix[:, 5].mean()),
        "per_class": {
            category_id: float(np.mean(values))
            for category_id, values in per_class.items()
        },
        "per_class_ap50": {
            category_id: float(values[0])
            for category_id, values in per_class.items()
        },
        "class_names": names,
        "f1_best": _at_grid(best_index, *macro),
        "f1_at": _at_grid(fixed_index, *macro),
        "per_class_f1_best": {
            category_id: _at_grid(int(np.argmax(curve[2])), *curve)
            for category_id, curve in curves.items()
        },
    }


def evaluate_scenarios(coco, detections, max_dets=500, score_thr=0.30,
                       scenarios=None):
    """Ayni tespitleri birden cok sinif tanimiyla degerlendirir.

    Model bir kez calisir, olcum dort farkli sekilde yapilir. Boylece "tek
    sinifa inersem ne kazanirim" sorusu yeniden egitim yapmadan cevaplanir.
    """
    results = {}
    for name, (group_map, class_names) in (scenarios or SCENARIOS).items():
        results[name] = evaluate_visdrone(
            coco, detections, max_dets=max_dets, group_map=group_map,
            class_names=class_names, score_thr=score_thr,
        )
    return results


def format_scenarios(results):
    """Senaryo karsilastirma tablosunu tek metne cevirir."""
    header = (
        f"{'senaryo':<22}{'AP':>8}{'AP50':>8}{'AP75':>8}"
        f"{'F1':>8}{'P':>8}{'R':>8}{'@conf':>7}"
    )
    lines = [header, "-" * len(header)]
    for name, metrics in results.items():
        best = metrics.get("f1_best")
        if best is None:
            lines.append(f"{name:<22}{'tespit yok':>47}")
            continue
        lines.append(
            f"{name:<22}{metrics['ap']:>8.4f}{metrics['ap50']:>8.4f}"
            f"{metrics['ap75']:>8.4f}{best['f1']:>8.4f}"
            f"{best['precision']:>8.4f}{best['recall']:>8.4f}"
            f"{best['score']:>7.2f}"
        )
    return "\n".join(lines)


def format_metrics(metrics, title="VisDrone"):
    """Degerlendirme ciktisini insan okunur tek bir metne cevirir."""
    names = metrics.get("class_names", {})
    lines = [
        f"{title}: AP@[.50:.95]={metrics['ap']:.4f}  "
        f"AP@0.50={metrics['ap50']:.4f}  AP@0.75={metrics['ap75']:.4f}",
    ]
    best, fixed = metrics.get("f1_best"), metrics.get("f1_at")
    if best and fixed:
        lines.append(
            f"  En iyi F1={best['f1']:.4f} (P={best['precision']:.4f} "
            f"R={best['recall']:.4f} @conf={best['score']:.2f})"
        )
        lines.append(
            f"  conf={fixed['score']:.2f}: F1={fixed['f1']:.4f} "
            f"P={fixed['precision']:.4f} R={fixed['recall']:.4f}"
        )
    if metrics.get("per_class"):
        lines.append(f"  {'sinif':<18}{'AP':>8}{'AP50':>8}{'F1':>8}")
        for category_id in sorted(metrics["per_class"]):
            f1 = metrics["per_class_f1_best"].get(category_id, {})
            lines.append(
                f"  {names.get(category_id, category_id):<18}"
                f"{metrics['per_class'][category_id]:>8.4f}"
                f"{metrics['per_class_ap50'][category_id]:>8.4f}"
                f"{f1.get('f1', float('nan')):>8.4f}"
            )
    return "\n".join(lines)


In [ ]:
%%writefile yolox_nano_visdrone.py
#!/usr/bin/env python3
"""KV260 DPU'suna (DPUCZDX8G) uyumlu YOLOX-Nano / VisDrone-DET deneyi.

DPU uyumlulugu icin iki degisiklik yapilir:
  1. act="relu": DPU, YOLOX'un varsayilan SiLU aktivasyonunu desteklemez.
  2. Focus stem'i DPUFocus ile degistirilir: Focus'un strided-slice islemi
     DPU'da calismaz; sabit one-hot agirlikli 2x2/stride-2 conv ayni
     space-to-depth dizilimini birebir uretir. Boylece model tek DPU
     subgraph'i olarak derlenir ve onceden egitilmis agirliklar gecerli kalir.
"""

import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

for _alias, _type in (("float", float), ("int", int), ("bool", bool)):
    if _alias not in np.__dict__:
        setattr(np, _alias, _type)

from yolox.data import COCODataset
from yolox.evaluators import COCOEvaluator
from yolox.exp import Exp as MyExp
from yolox.utils import is_main_process

for _helper_dir in (Path(__file__).resolve().parent,
                    Path(__file__).resolve().parent.parent):
    if str(_helper_dir) not in sys.path:
        sys.path.insert(0, str(_helper_dir))
from visdrone_eval import evaluate_visdrone, format_metrics  # noqa: E402

#: Saha tanimi: yalnizca insan ve tasit. `vehicle` bisiklet/motosiklet dahil
#: her turlu tasiti kapsar. Veri `visdrone2coco.py --classes 2` ile uretilir.
TARGET_CLASSES = ("person", "vehicle")


def assert_class_scheme(coco, expected):
    """Anotasyon semasi modelin sinif sayisiyla uyusmuyorsa hemen durur.

    Eski 10 sinifli `instances_*.json` diskte kalirsa 2 sinifli bir bas
    sessizce yanlis etiketlerle egitilir ve bu ancak saatler sonra
    degerlendirmede fark edilir. Ucuz bir kapi, pahali bir hatayi onler.
    """
    if expected is None:
        return
    found = sorted(coco.cats)
    if found != list(range(1, expected + 1)):
        names = [coco.cats[c].get("name", c) for c in found]
        raise ValueError(
            f"Anotasyon semasi uyusmuyor: {expected} sinif bekleniyordu, "
            f"{len(found)} bulundu ({names}). Veriyi "
            f"'visdrone2coco.py --classes {expected}' ile yeniden uretin."
        )


class VisDroneTrainingDataset(COCODataset):
    """Egitimde bilincli olarak ignore edilen alanlari negatif ornek yapmaz.

    Donusturucu genel bolgeleri image.ignore_regions, score=0 kutularini ise
    ``iscrowd=1`` olarak saklar. YOLOX crowd kutularini hedeflerden cikartir;
    burada ek olarak ilgili pikseller 114 ile maskelenir.
    """

    def __init__(self, *args, expected_num_classes=None, **kwargs):
        if kwargs.get("cache", False):
            raise ValueError("Ignore maskeleme ile --cache kullanmayin.")
        super().__init__(*args, **kwargs)
        assert_class_scheme(self.coco, expected_num_classes)
        self._collect_ignore_boxes()

    def _collect_ignore_boxes(self):
        self.ignore_boxes = {}
        for image_id in self.ids:
            ann_ids = self.coco.getAnnIds(imgIds=[int(image_id)], iscrowd=True)
            boxes = {
                tuple(int(round(v)) for v in ann["bbox"])
                for ann in self.coco.loadAnns(ann_ids)
            }
            boxes.update(
                tuple(int(round(v)) for v in bbox)
                for bbox in self.coco.imgs[int(image_id)].get("ignore_regions", ())
            )
            self.ignore_boxes[int(image_id)] = sorted(boxes)

    def load_image(self, index):
        img = super().load_image(index)
        height, width = img.shape[:2]
        for x, y, w, h in self.ignore_boxes.get(int(self.ids[index]), ()):
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(width, x + w), min(height, y + h)
            if x2 > x1 and y2 > y1:
                img[y1:y2, x1:x2] = 114
        return img


class VisDroneEvaluator(COCOEvaluator):
    """Resmi DET toolkit eslestirme/VOC AP mantigiyla VisDrone AP@500."""

    def evaluate_prediction(self, data_dict, statistics):
        if not is_main_process():
            return 0, 0, None
        if not data_dict:
            return 0, 0, "VisDrone AP@500: hic tespit yok\n"

        coco_gt = self.dataloader.dataset.coco
        # Sinif adlari COCO kategorilerinden okunur: veri semasi degisirse
        # tablo da kendiliginden dogru kalir.
        metrics = evaluate_visdrone(
            coco_gt, data_dict, max_dets=500,
            num_classes=self.num_classes, score_thr=self.deploy_conf,
        )
        ap, ap50 = metrics["ap"], metrics["ap50"]

        inference_time, nms_time, n_samples = (v.item() for v in statistics)
        batch = self.dataloader.batch_size
        info = (
            "VisDrone DET-style evaluation (ignore, global maxDets=500)\n"
            + format_metrics(metrics, "saha") + "\n"
            + f"Average forward = {1000 * inference_time / (n_samples * batch):.2f} ms, "
            + f"NMS = {1000 * nms_time / (n_samples * batch):.2f} ms\n"
        )
        return ap, ap50, info


class DPUFocus(nn.Module):
    """YOLOX Focus katmaninin DPU-uyumlu birebir karsiligi.

    Focus'un dilimleme + birlestirme sirasi (TL, BL, TR, BR) sabit agirlikli
    bir conv ile ayni matematikle uretilir. `conv` alt modul adi korundugu
    icin onceden egitilmis Focus agirliklari dogrudan yuklenebilir.
    """

    def __init__(self, in_channels, out_channels, ksize=1, stride=1, act="silu"):
        super().__init__()
        from yolox.models.network_blocks import BaseConv

        self.space_to_depth = nn.Conv2d(
            in_channels, in_channels * 4, kernel_size=2, stride=2, bias=False
        )
        w = torch.zeros(in_channels * 4, in_channels, 2, 2)
        # Focus birlestirme sirasi: TL=(0,0), BL=(1,0), TR=(0,1), BR=(1,1)
        for block, (r, c) in enumerate(((0, 0), (1, 0), (0, 1), (1, 1))):
            for ch in range(in_channels):
                w[block * in_channels + ch, ch, r, c] = 1.0
        with torch.no_grad():
            self.space_to_depth.weight.copy_(w)
        self.space_to_depth.weight.requires_grad = False
        self.conv = BaseConv(in_channels * 4, out_channels, ksize, stride, act=act)

    def forward(self, x):
        return self.conv(self.space_to_depth(x))


class Exp(MyExp):
    def __init__(self):
        super().__init__()
        # ---- model: YOLOX-Nano geometrisi, DPU-uyumlu aktivasyon ----
        self.depth = 0.33
        self.width = 0.25
        self.act = "relu"
        self.num_classes = len(TARGET_CLASSES)

        # ---- girdi boyutu: kaynak videonun en-boy oranina uydurulmus ----
        # 1920x1080'i 640x640'a letterbox etmek kanvasin %44'unu gri dolguya
        # harcar ve etkin olcek 0.333'te kalir. 896x512 (16:9) ayni kareyi
        # 0.467 olcekle isler: %12 daha fazla hesapla %40 daha yuksek
        # cozunurluk. Kucuk nesne recall'unun ana kaldiraci budur.
        # (h, w) sirasi YOLOX kuralidir.
        self.input_size = (512, 896)
        self.test_size = (512, 896)
        # random_resize yuksekligi 32*s, genisligi 32*int(s*896/512) yapar;
        # s in [14, 18] -> 448x768 .. 576x992 araligi.
        self.random_size = (14, 18)

        # ---- veri seti ----
        self.data_dir = "datasets/visdrone_coco"
        self.train_ann = "instances_train.json"
        self.val_ann = "instances_val.json"
        self.data_num_workers = 2
        self.dataset = None

        # ---- augmentasyon (nano varsayilanlari) ----
        self.mosaic_prob = 0.5
        self.mosaic_scale = (0.5, 1.5)
        self.enable_mixup = False

        # ---- egitim ----
        # 10 sinifli calistirmada model epoch ~40'ta doymustu (epoch 45 -> 80
        # arasi AP@0.50 yalnizca +0.006). 40 epoch ayni sonucu yari surede
        # verir. Son 10 epoch mosaic kapali + L1 loss acik: kutu hassasiyetini
        # iyilestirdigi icin (asil darbogazimiz) bu faz korunuyor.
        self.max_epoch = 40
        self.warmup_epochs = 5
        self.no_aug_epochs = 10
        self.eval_interval = 5

        # ---- dagitim calisma noktasi ----
        # Kartta dusuk esikle calisip false positive'leri tracker'in "N karede
        # gorunmeli" kurali ile eleyecegiz (ByteTrack mantigi). Degerlendirme
        # F1'i bu esikte de raporlar.
        self.deploy_conf = 0.15
        self.print_interval = 50
        self.save_history_ckpt = False  # Kaggle diskini doldurmamak icin

        self.exp_name = os.path.split(os.path.realpath(__file__))[1].split(".")[0]

    def get_model(self, sublinear=False):
        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03

        if "model" not in self.__dict__:
            from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead

            in_channels = [256, 512, 1024]
            # Nano: depthwise=True
            backbone = YOLOPAFPN(
                self.depth, self.width, in_channels=in_channels,
                act=self.act, depthwise=True,
            )
            head = YOLOXHead(
                self.num_classes, self.width, in_channels=in_channels,
                act=self.act, depthwise=True,
            )
            self.model = YOLOX(backbone, head)
            # DPU: slice tabanli Focus stem'i esdeger conv surumuyle degistir
            self.model.backbone.backbone.stem = DPUFocus(
                3, int(self.width * 64), ksize=3, act=self.act
            )

        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)
        return self.model

    def get_dataset(self, cache=False, cache_type="ram"):
        from yolox.data import TrainTransform

        return VisDroneTrainingDataset(
            data_dir=self.data_dir,
            json_file=self.train_ann,
            name="train_images",
            img_size=self.input_size,
            expected_num_classes=self.num_classes,
            # VisDrone karelerinde yuzlerce nesne olabilir; varsayilan 50 cok dusuk
            preproc=TrainTransform(
                max_labels=1000, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob
            ),
            cache=cache,
            cache_type=cache_type,
        )

    def get_data_loader(self, batch_size, is_distributed, no_aug=False, cache_img=None):
        # yolox_base.Exp.get_data_loader kopyasi; tek fark mozaik donusumunde
        # max_labels=4000 (4 yogun VisDrone karesinin mozaigini kirpmaz).
        import torch.distributed as dist

        from yolox.data import (
            TrainTransform,
            YoloBatchSampler,
            DataLoader,
            InfiniteSampler,
            MosaicDetection,
            worker_init_reset_seed,
        )
        from yolox.utils import wait_for_the_master

        if self.dataset is None:
            with wait_for_the_master():
                assert cache_img is None, (
                    "cache_img must be None if you didn't create self.dataset before launch"
                )
                self.dataset = self.get_dataset(cache=False, cache_type=cache_img)

        self.dataset = MosaicDetection(
            dataset=self.dataset,
            mosaic=not no_aug,
            img_size=self.input_size,
            preproc=TrainTransform(
                max_labels=4000, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob
            ),
            degrees=self.degrees,
            translate=self.translate,
            mosaic_scale=self.mosaic_scale,
            mixup_scale=self.mixup_scale,
            shear=self.shear,
            enable_mixup=self.enable_mixup,
            mosaic_prob=self.mosaic_prob,
            mixup_prob=self.mixup_prob,
        )

        if is_distributed:
            batch_size = batch_size // dist.get_world_size()

        sampler = InfiniteSampler(len(self.dataset), seed=self.seed if self.seed else 0)
        batch_sampler = YoloBatchSampler(
            sampler=sampler,
            batch_size=batch_size,
            drop_last=False,
            mosaic=not no_aug,
        )
        dataloader_kwargs = {
            "num_workers": self.data_num_workers,
            "pin_memory": True,
            "batch_sampler": batch_sampler,
            "worker_init_fn": worker_init_reset_seed,
        }
        return DataLoader(self.dataset, **dataloader_kwargs)

    def get_eval_dataset(self, **kwargs):
        from yolox.data import COCODataset, ValTransform

        legacy = kwargs.get("legacy", False)
        dataset = COCODataset(
            data_dir=self.data_dir,
            json_file=self.val_ann,
            name="val_images",
            img_size=self.test_size,
            preproc=ValTransform(legacy=legacy),
        )
        assert_class_scheme(dataset.coco, self.num_classes)
        return dataset

    def get_evaluator(self, batch_size, is_distributed, testdev=False, legacy=False):
        return VisDroneEvaluator(
            dataloader=self.get_eval_loader(
                batch_size, is_distributed, testdev=testdev, legacy=legacy
            ),
            img_size=self.test_size,
            confthre=self.test_conf,
            nmsthre=self.nmsthre,
            num_classes=self.num_classes,
            testdev=testdev,
        )


In [ ]:
# Baslangic agirligi: YOLOX'un resmi Megvii COCO checkpoint'i.
# Vitis AI 3.5 Model Zoo paketi kullanilmaz; arac zinciri Vitis AI 3.0'da kalir.
import importlib.util
import urllib.request

WDIR = Path(WORK) / "weights"
WDIR.mkdir(exist_ok=True)

spec = importlib.util.spec_from_file_location("exp_mod", "yolox_nano_visdrone.py")
exp_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exp_mod)
ref_state = exp_mod.Exp().get_model().state_dict()

mg = WDIR / "yolox_nano.pth"
if not mg.exists():
    urllib.request.urlretrieve(
        "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth",
        mg,
    )

# PyTorch 2.6+ varsayilani weights_only=True, Vitis AI 3.0/PyTorch 1.12 ise
# bu parametreyi tanimaz. Iki ortamda da acik ve guvenilir yerel checkpoint yukle.
try:
    raw = torch.load(mg, map_location="cpu", weights_only=False)
except TypeError:
    raw = torch.load(mg, map_location="cpu")
inner = raw.get("model", raw.get("state_dict", raw)) if isinstance(raw, dict) else raw
inner = {(k[7:] if k.startswith("module.") else k): v for k, v in inner.items()}

matched = {
    k: v for k, v in inner.items()
    if k in ref_state and ref_state[k].shape == v.shape
}
missing = sorted(set(ref_state) - set(matched))
allowed_missing = all(
    k == "backbone.backbone.stem.space_to_depth.weight" or ".cls_preds." in k
    for k in missing
)
ratio = len(matched) / max(len(ref_state), 1)
assert ratio >= 0.95 and allowed_missing, (
    f"Baslangic checkpoint'i mimariyle uyumsuz: eslesme={ratio:.1%}, "
    f"beklenmeyen eksikler={missing}"
)

# Tam ve sekil-uyumlu bir state_dict yaz. 2 sinifli head ve DPUFocus agirligi
# yeni modelin baslangicindan, diger katmanlar Megvii'den gelir.
init_state = dict(ref_state)
init_state.update(matched)
INIT_CKPT = str(WDIR / "init_ckpt.pth")
torch.save({
    "model": init_state,
    "meta": {
        "source": str(mg),
        "yolox_commit": YOLOX_COMMIT,
        "matched_ratio": ratio,
    },
}, INIT_CKPT)
print(f"Megvii baslangici dogrulandi: {ratio:.1%} -> {INIT_CKPT}")
print("Yalnizca cls head (2 sinif) ve sabit DPUFocus katmani yeniden baslatildi.")

## Eğitim

- T4 GPU'da 640×640 nano için 80 epoch yaklaşık **3-6 saat** sürer.
- GPU belleği yetmezse `BATCH` değerini 8'e düşürün.
- Checkpoint'ler `YOLOX_outputs/yolox_nano_visdrone/` altına yazılır: `latest_ckpt.pth` her epoch sonunda güncellenir, `best_ckpt.pth` en iyi val AP@500 değerine aittir.
- Oturum koparsa: bu not defterinin Output'unu yeni oturuma input olarak bağlayıp *Devam (resume)* hücresini kullanın.

In [ ]:
# 2 sinif (person/vehicle), girdi 896x512 (16:9), 40 epoch.
# Beklenen sure T4'te ~1.5-2 saat; 10 sinifli 640x640 kosumunda model
# epoch ~40'ta doymustu, bu yuzden 80 yerine 40 epoch kullaniliyor.
BATCH = 16  # OOM olursa 8 yapin

%cd {WORK}
!python YOLOX/tools/train.py -f yolox_nano_visdrone.py -d 1 -b {BATCH} --fp16 -c weights/init_ckpt.pth


In [ ]:
# Devam (resume): onceki oturumun Output'unu bu oturuma input olarak bagladiktan
# sonra asagidaki degiskene latest_ckpt.pth yolunu yazip calistirin.
RESUME_CKPT = ""  # ornek: "/kaggle/input/ONCEKI-NOTEBOOK/YOLOX_outputs/yolox_nano_visdrone/latest_ckpt.pth"

if RESUME_CKPT:
    !python YOLOX/tools/train.py -f yolox_nano_visdrone.py -d 1 -b {BATCH} --fp16 --resume -c "{RESUME_CKPT}"
else:
    print("RESUME_CKPT bos; bu hucre yalnizca yarim kalan egitimi surdurmek icindir.")

In [ ]:
# VisDrone val: resmi ignore filtresi + global top-500 + VOC AP mantigi.
# Ayrica sinif bazli AP ve P/R/F1 ile 3-sinif (person/twowheeler/vehicle)
# gruplanmis olcum basilir. Bu sayilari not edin: kuantalama sonrasi ayni
# protokolle karsilastirilacak.


def find_best_ckpt():
    """Bu oturumda egitildiyse yerelden, aksi halde bagli input'tan alir."""
    local = Path("YOLOX_outputs/yolox_nano_visdrone/best_ckpt.pth")
    if local.exists():
        return local
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for candidate in sorted(input_root.glob("**/best_ckpt.pth")):
            return candidate
    raise SystemExit(
        "best_ckpt.pth bulunamadi. Once egitim hucresini calistirin veya "
        "onceki oturumun ciktisini bu oturuma input olarak baglayin."
    )


BEST_CKPT = str(find_best_ckpt())
print("checkpoint:", BEST_CKPT)
!python YOLOX/tools/eval.py -f yolox_nano_visdrone.py -c "{BEST_CKPT}" -d 1 -b 16 --conf 0.001


In [ ]:
%%writefile tiling.py
#!/usr/bin/env python3
"""Dilimlenmis (tiled) cikarim icin saf geometri ve birlestirme mantigi.

Neden: 1920x1080 bir kareyi 640x640'a kucultmek olcegi ~0.33'e dusurur, yani
20 piksellik bir yaya 7 piksele iner ve stride-8 haritasinda tek hucreye
sikisir. Kareyi ortusen parcalara bolup her parcayi ayri ayri modele vermek
ayni nesneyi ~2 kat buyuk tutar. SAHI calismasi VisDrone'da bu teknigin
+5..7 AP getirdigini olcmustur (arXiv 2202.06934).

Bu modul bilincli olarak torch/cv2 icermez: ayni mantik hem Kaggle'daki
Python degerlendirmesinde hem de KV260'taki C++ uygulamasinda kullanilacak,
o yuzden once burada saf ve test edilebilir halde durur.

Ortusme kurali: parca kenarina denk gelen nesne iki parcada da **tam olarak**
gorunsun diye ortusme payi en buyuk nesneden genis secilmelidir. VisDrone'da
nesneler <50 px oldugundan varsayilan %20 ortusme (1920 genislikte ~192 px)
fazlasiyla yeterlidir; boylece parca sinirinda ikiye bolunmus yarim kutular
olusmaz ve NMS kopyalari sorunsuz birlestirir.
"""

import numpy as np


def tile_rects(width, height, cols=2, rows=2, overlap=0.2, include_full=True):
    """Ortusen parca dikdortgenlerini `(x0, y0, x1, y1)` olarak uretir.

    Parcalar esit araliklarla yerlestirilir ve her biri temel hucre boyutunun
    `(1 + overlap)` katidir. `include_full` ile tum kare de bir "parca" olarak
    listeye eklenir: kucuk nesneleri parcalar, buyuk nesneleri tam kare yakalar.

    Donen dikdortgenler tamsayidir ve goruntu sinirlari icinde kalir.
    """
    if width <= 0 or height <= 0:
        raise ValueError("gecersiz goruntu boyutu")
    if cols < 1 or rows < 1:
        raise ValueError("cols ve rows en az 1 olmali")
    if not 0.0 <= overlap < 1.0:
        raise ValueError("overlap [0, 1) araliginda olmali")

    def axis_positions(total, count):
        base = total / count
        size = min(float(total), base * (1.0 + overlap))
        starts = []
        for index in range(count):
            center = (index + 0.5) * base
            start = min(max(center - size / 2.0, 0.0), total - size)
            starts.append(start)
        return size, starts

    tile_w, xs = axis_positions(width, cols)
    tile_h, ys = axis_positions(height, rows)

    rects = []
    for y0 in ys:
        for x0 in xs:
            x_start, y_start = int(round(x0)), int(round(y0))
            x_end = min(width, x_start + int(round(tile_w)))
            y_end = min(height, y_start + int(round(tile_h)))
            rects.append((x_start, y_start, x_end, y_end))

    if include_full and (cols > 1 or rows > 1):
        rects.append((0, 0, int(width), int(height)))
    # Ayni dikdortgen birden fazla kez uretilmisse (ornegin 1x1) tekrari at.
    unique = []
    for rect in rects:
        if rect not in unique:
            unique.append(rect)
    return unique


def offset_boxes(boxes, rect):
    """Parca-yerel xyxy kutularini tam kare koordinatlarina tasir."""
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    if boxes.size == 0:
        return boxes
    shifted = boxes.copy()
    shifted[:, [0, 2]] += float(rect[0])
    shifted[:, [1, 3]] += float(rect[1])
    return shifted


def nms_xyxy(boxes, scores, iou_thr=0.45):
    """Tek sinif icin standart greedy NMS; kalan kutularin indekslerini doner."""
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    if boxes.shape[0] == 0:
        return np.empty(0, dtype=np.int64)

    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    areas = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    order = scores.argsort()[::-1]

    keep = []
    while order.size > 0:
        current = order[0]
        keep.append(current)
        if order.size == 1:
            break
        rest = order[1:]
        inter_w = np.maximum(
            0.0, np.minimum(x2[current], x2[rest]) - np.maximum(x1[current], x1[rest])
        )
        inter_h = np.maximum(
            0.0, np.minimum(y2[current], y2[rest]) - np.maximum(y1[current], y1[rest])
        )
        inter = inter_w * inter_h
        union = areas[current] + areas[rest] - inter
        iou = np.where(union > 0, inter / np.maximum(union, 1e-12), 0.0)
        order = rest[iou <= iou_thr]
    return np.asarray(keep, dtype=np.int64)


def merge_tiled(tile_results, iou_thr=0.45, max_dets=500):
    """Parca sonuclarini tam kare koordinatlarinda birlestirir.

    `tile_results`: `(rect, boxes_xyxy, scores, class_ids)` dizisi. Kutular
    parca-yerel koordinatlardadir (letterbox tersi zaten uygulanmis olmali).

    NMS **sinif bazinda** uygulanir: ust uste binen bir yaya ile bisiklet
    birbirini elemez. Ayni nesnenin komsu parcalardaki kopyalari ise ayni
    sinifta olduklari icin birlesir.

    Doner: `(boxes, scores, class_ids)`, skora gore azalan sirali.
    """
    all_boxes, all_scores, all_classes = [], [], []
    for rect, boxes, scores, class_ids in tile_results:
        boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
        if boxes.shape[0] == 0:
            continue
        all_boxes.append(offset_boxes(boxes, rect))
        all_scores.append(np.asarray(scores, dtype=np.float64).reshape(-1))
        all_classes.append(np.asarray(class_ids).reshape(-1))

    if not all_boxes:
        return (
            np.zeros((0, 4), dtype=np.float64),
            np.zeros(0, dtype=np.float64),
            np.zeros(0, dtype=np.int64),
        )

    boxes = np.concatenate(all_boxes, axis=0)
    scores = np.concatenate(all_scores, axis=0)
    classes = np.concatenate(all_classes, axis=0)

    keep = []
    for class_id in np.unique(classes):
        mask = np.nonzero(classes == class_id)[0]
        kept = nms_xyxy(boxes[mask], scores[mask], iou_thr=iou_thr)
        keep.extend(mask[kept].tolist())

    keep = np.asarray(keep, dtype=np.int64)
    order = keep[scores[keep].argsort()[::-1]][:max_dets]
    return boxes[order], scores[order], classes[order]


In [ ]:
%%writefile eval_tiled.py
#!/usr/bin/env python3
"""Tam-kare ve dilimlenmis (tiled) cikarimi ayni val setinde karsilastirir.

Amac: tiling'in VisDrone'da kac AP/F1 puani getirdigini **olcmek**. Kazanc
olculmeden KV260 tarafina C++ yazmak korlemesine is olur; oradaki bedel
kare basina DPU cagrisinin 1'den 5'e cikmasidir.

Kullanim (Kaggle, egitim bittikten sonra):
    python eval_tiled.py --exp-file yolox_nano_visdrone.py \\
        --ckpt YOLOX_outputs/yolox_nano_visdrone/best_ckpt.pth \\
        --data-dir datasets/visdrone_coco --grid 2x2 --overlap 0.2
"""

import argparse
import importlib.util
import json
import time
from pathlib import Path

import numpy as np
import torch

from visdrone_eval import evaluate_visdrone, format_metrics

try:  # tiling.py notebook'ta yan yana yazilir, repoda tools/ altindadir
    from tiling import merge_tiled, tile_rects
except ImportError:  # pragma: no cover
    from tools.tiling import merge_tiled, tile_rects


def load_exp(exp_file):
    spec = importlib.util.spec_from_file_location("exp_module", exp_file)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.Exp()


def letterbox(image, size):
    """YOLOX ValTransform ile ayni: en-boy korunur, sol-ust, 114 dolgu."""
    import cv2

    padded = np.full((size, size, 3), 114, dtype=np.uint8)
    ratio = min(size / image.shape[0], size / image.shape[1])
    new_h, new_w = int(image.shape[0] * ratio), int(image.shape[1] * ratio)
    if new_h > 0 and new_w > 0:
        padded[:new_h, :new_w] = cv2.resize(
            image, (new_w, new_h), interpolation=cv2.INTER_LINEAR
        )
    return padded.transpose(2, 0, 1).astype(np.float32), ratio


@torch.no_grad()
def infer_tiles(model, image, rects, size, conf_thr, nms_thr, num_classes, device):
    """Her parcayi modele verir; parca-yerel xyxy tespitlerini dondurur."""
    from yolox.utils import postprocess

    batch, ratios = [], []
    for x0, y0, x1, y1 in rects:
        tensor, ratio = letterbox(image[y0:y1, x0:x1], size)
        batch.append(tensor)
        ratios.append(ratio)

    outputs = postprocess(
        model(torch.from_numpy(np.stack(batch)).to(device)),
        num_classes, conf_thr, nms_thr, class_agnostic=False,
    )

    results = []
    for rect, ratio, output in zip(rects, ratios, outputs):
        if output is None or len(output) == 0:
            results.append((rect, np.zeros((0, 4)), [], []))
            continue
        output = output.cpu().numpy()
        boxes = output[:, 0:4] / ratio          # letterbox tersi -> parca-yerel
        scores = output[:, 4] * output[:, 5]    # objectness * sinif skoru
        classes = output[:, 6].astype(np.int64)
        results.append((rect, boxes, scores, classes))
    return results


def run(args):
    import cv2
    from pycocotools.coco import COCO

    device = "cuda" if torch.cuda.is_available() else "cpu"
    exp = load_exp(args.exp_file)
    size = exp.test_size[0]

    model = exp.get_model().to(device).eval()
    checkpoint = torch.load(args.ckpt, map_location="cpu")
    model.load_state_dict(checkpoint.get("model", checkpoint))

    data_dir = Path(args.data_dir)
    coco = COCO(str(data_dir / "annotations" / args.val_ann))
    image_dir = data_dir / args.image_folder

    cols, rows = (int(v) for v in args.grid.lower().split("x"))
    configs = {
        "tam kare": (1, 1, False),
        f"tiled {args.grid}": (cols, rows, args.include_full),
    }

    all_metrics = {}
    for label, (grid_cols, grid_rows, include_full) in configs.items():
        detections = []
        started = time.time()
        tile_total = 0
        for image_id in coco.getImgIds():
            info = coco.imgs[image_id]
            image = cv2.imread(str(image_dir / info["file_name"]))
            if image is None:
                raise SystemExit(f"goruntu okunamadi: {info['file_name']}")

            rects = tile_rects(
                info["width"], info["height"], grid_cols, grid_rows,
                overlap=args.overlap, include_full=include_full,
            )
            tile_total += len(rects)
            tile_results = infer_tiles(
                model, image, rects, size, args.conf, args.nms,
                exp.num_classes, device,
            )
            boxes, scores, classes = merge_tiled(
                tile_results, iou_thr=args.nms, max_dets=args.max_dets
            )
            for box, score, class_id in zip(boxes, scores, classes):
                detections.append({
                    "image_id": int(image_id),
                    "category_id": int(class_id) + 1,
                    "bbox": [
                        float(box[0]), float(box[1]),
                        float(box[2] - box[0]), float(box[3] - box[1]),
                    ],
                    "score": float(score),
                })

        elapsed = time.time() - started
        # Sinif semasi COCO'dan okunur; 2 sinifli veride de 10 sinifli veride
        # de dogru calisir. Sinif gruplama burada yapilmaz - bu scriptin isi
        # yalnizca tam kare ile dilimlenmis cikarimi karsilastirmak.
        metrics = evaluate_visdrone(
            coco, detections, max_dets=args.max_dets,
            num_classes=exp.num_classes, score_thr=args.score_thr,
        )
        all_metrics[label] = metrics

        print(f"\n{'=' * 66}\n{label}  "
              f"({tile_total / max(1, len(coco.getImgIds())):.1f} parca/kare, "
              f"{elapsed:.0f} s, {len(detections)} tespit)\n{'=' * 66}")
        print(format_metrics(metrics, label))

        if args.save_json:
            out = Path(args.save_json).with_suffix("")
            slug = label.replace(" ", "_").replace("/", "")
            Path(f"{out}_{slug}.json").write_text(json.dumps(detections))

    print(f"\n{'=' * 66}\nKARSILASTIRMA\n{'=' * 66}")
    print(f"{'yapilandirma':<20}{'AP50':>9}{'AP':>9}{'AP75':>9}"
          f"{'F1':>9}{'R':>9}")
    for label, m in all_metrics.items():
        print(f"{label:<20}{m['ap50']:>9.4f}{m['ap']:>9.4f}{m['ap75']:>9.4f}"
              f"{m['f1_best']['f1']:>9.4f}{m['f1_best']['recall']:>9.4f}")

    labels = list(all_metrics)
    if len(labels) == 2:
        base, tiled = all_metrics[labels[0]], all_metrics[labels[1]]
        print(f"\ntiling kazanci: AP50 {tiled['ap50'] - base['ap50']:+.4f}, "
              f"AP {tiled['ap'] - base['ap']:+.4f}, "
              f"F1 {tiled['f1_best']['f1'] - base['f1_best']['f1']:+.4f}, "
              f"recall {tiled['f1_best']['recall'] - base['f1_best']['recall']:+.4f}")
        print("Karar olcutu: bu kazanc, kare basina ~5 kat DPU maliyetini "
              "ve dusen FPS'i hakli cikariyor mu?")


def parse_args():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--exp-file", required=True)
    p.add_argument("--ckpt", required=True)
    p.add_argument("--data-dir", default="datasets/visdrone_coco")
    p.add_argument("--val-ann", default="instances_val.json")
    p.add_argument("--image-folder", default="val_images")
    p.add_argument("--grid", default="2x2", help="parca izgarasi, or. 2x2 / 3x2")
    p.add_argument("--overlap", type=float, default=0.2)
    p.add_argument("--include-full", action="store_true", default=True,
                   help="parcalarin yaninda tum kareyi de isle (buyuk nesneler)")
    p.add_argument("--no-include-full", dest="include_full",
                   action="store_false")
    p.add_argument("--conf", type=float, default=0.001,
                   help="AP olcumu icin dusuk tutulur; F1 egrisi zaten taranir")
    p.add_argument("--score-thr", type=float, default=0.15,
                   help="F1'in ayrica raporlanacagi dagitim esigi (kartla ayni)")
    p.add_argument("--nms", type=float, default=0.65)
    p.add_argument("--max-dets", type=int, default=500)
    p.add_argument("--save-json", default=None)
    return p.parse_args()


if __name__ == "__main__":
    run(parse_args())


In [ ]:
# Tiling kazancini OLC: tam kare cikarim vs 2x2 parcali cikarim, ayni val seti.
# Model yeniden egitilmez - tiling tamamen cikarim zamani teknigidir.
# Cikti: iki yapilandirmanin AP/F1 tablosu + fark satiri.
!python eval_tiled.py --exp-file yolox_nano_visdrone.py     --ckpt "{BEST_CKPT}" --data-dir "{DATASET_DIR}"     --grid 2x2 --overlap 0.2 --save-json detections


In [ ]:
# Gorsel kontrol: kutular + MERKEZ NOKTALARI (cx, cy)
# KV260 uygulamasindaki hesabin aynisi: cx=(x1+x2)/2, cy=(y1+y2)/2
import cv2
import importlib.util
import matplotlib.pyplot as plt

from yolox.data.data_augment import ValTransform
from yolox.utils import postprocess

spec = importlib.util.spec_from_file_location("exp_mod2", "yolox_nano_visdrone.py")
exp_mod2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exp_mod2)
exp = exp_mod2.Exp()
CLASSES = exp_mod2.TARGET_CLASSES

model = exp.get_model().eval()
try:
    ckpt = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
except TypeError:
    ckpt = torch.load(BEST_CKPT, map_location="cpu")
model.load_state_dict(ckpt["model"], strict=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

CONF_VIS = 0.15  # kartla ayni dagitim esigi
val_transform = ValTransform(legacy=False)
sample_paths = sorted((DATASET_DIR / "val_images").glob("*.jpg"))[:3]

fig, axes = plt.subplots(len(sample_paths), 1, figsize=(16, 9 * len(sample_paths)))
axes = np.atleast_1d(axes)
for ax, p in zip(axes, sample_paths):
    img0 = cv2.imread(str(p))
    h0, w0 = img0.shape[:2]
    ratio = min(exp.test_size[0] / h0, exp.test_size[1] / w0)
    timg, _ = val_transform(img0, None, exp.test_size)
    tin = torch.from_numpy(timg).unsqueeze(0).float().to(device)
    with torch.no_grad():
        outputs = postprocess(model(tin), exp.num_classes, CONF_VIS, exp.nmsthre)
    vis = img0.copy()
    det = outputs[0]
    n = 0
    if det is not None:
        for x1, y1, x2, y2, obj_conf, cls_conf, cls_id in det.cpu().numpy():
            x1, y1, x2, y2 = x1 / ratio, y1 / ratio, x2 / ratio, y2 / ratio
            cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0  # MERKEZ NOKTASI
            score = obj_conf * cls_conf
            cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            cv2.circle(vis, (int(cx), int(cy)), 4, (0, 0, 255), -1)
            cv2.putText(vis, f"{CLASSES[int(cls_id)]} {score:.2f} ({int(cx)},{int(cy)})",
                        (int(x1), max(12, int(y1) - 4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
            n += 1
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{p.name} - {n} tespit (kirmizi nokta = merkez)")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Oracle VM'e tasinacak dosyalari paketle
import shutil

ART = Path(WORK) / "artifacts"
ART.mkdir(exist_ok=True)
shutil.copy(BEST_CKPT, ART / "best_ckpt.pth")
shutil.copy("yolox_nano_visdrone.py", ART / "yolox_nano_visdrone.py")
shutil.copy("visdrone_eval.py", ART / "visdrone_eval.py")
for _extra in ("tiling.py", "eval_tiled.py"):
    shutil.copy(_extra, ART / _extra)
(ART / "classes.txt").write_text("\n".join(CLASSES))
(ART / "YOLOX_COMMIT.txt").write_text(YOLOX_COMMIT + "\n")
zip_path = shutil.make_archive(str(Path(WORK) / "yolox_visdrone_artifacts"), "zip", ART)
print("Hazir:", zip_path)
print("Not defteri kaydedilince bu dosyayi Output sekmesinden indirebilirsiniz.")

## Sonraki adım: Oracle VM'de kuantalama

1. `yolox_visdrone_artifacts.zip` dosyasını indirin (`best_ckpt.pth` + exp + `visdrone_eval.py` + sınıf listesi + `YOLOX_COMMIT.txt`).
2. VisDrone **val** klasörünü (548 görüntü; resmi ignore filtresi ve AP@500 testi için) ve tercihen ~300 train görüntüsünü VM'e kopyalayın.
3. Projedeki `quantize/README.md` adımlarını izleyin: aynı YOLOX commit'i → Vitis AI 3.0 PTQ → INT8 AP@500 kayıp kapısı → xmodel export → yalnızca tek DPU subgraph kontrolü.